# Lecture 0.1 — Accessing course data

**ICAT3370 Foundations of Space Data and Earth Observation Systems**

In this course you will read satellite data from three different kinds of places.
Understanding the difference is itself part of the course content: modern Earth
Observation is as much about *data infrastructure* as about sensors.

| Route | What it is | When to use |
|---|---|---|
| **1. Shared project disk** | Files on the supercomputer's filesystem | Fastest — default in exercise sessions |
| **2. Object storage (Allas)** | Files served over HTTPS from CSC's object store | Works from anywhere — also outside Roihu |
| **3. STAC catalog** | A searchable index of a global satellite archive | When you need to *find* data, not just read it |

All three deliver the same kind of file: a **Cloud-Optimized GeoTIFF (COG)** —
a raster arranged so that software can read only the parts it needs
(a window, a coarse overview) instead of downloading everything.

## Setup

One environment setting first. When GDAL (the library underneath everything geospatial)
opens a remote file, it also probes for possible *sidecar* metadata files next to it.
Our COGs are self-contained, so the probing only produces harmless 403 warnings.
This line disables it — and makes remote opens faster:

In [ ]:
import os
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

import rioxarray
import matplotlib.pyplot as plt
print("ready")

## Route 1 — Shared project disk (`/scratch`)

The course keeps working copies of the exercise data on the project's shared disk.
It is mounted on every session, read-only for you, and reading it is as fast as disk I/O gets.

The Vaasa scene we use below is a **Sentinel-2 L2A** acquisition from **16 July 2026**,
0 % cloud cover, cropped to the Vaasa / Kvarken area.

In [ ]:
DATA = "/scratch/project_2020353/data/vaasa_s2"

visual = rioxarray.open_rasterio(f"{DATA}/vaasa_20260716_visual.tif")
print("shape:", visual.shape, "  (bands, rows, cols)")

visual.plot.imshow(figsize=(7, 7))
plt.title("True color — read from /scratch")
plt.show()

## Route 2 — Object storage (CSC Allas)

The *master copies* of the course data live in **Allas**, CSC's object storage —
the same kind of system as Amazon S3. Objects in a public bucket are readable by
anyone through a plain HTTPS URL. This means the exact same code works on Roihu,
on your laptop, or anywhere with internet.

Note two practical details in the code below, both classics:

- `.squeeze()` — single-band rasters open with a size-1 *band* dimension;
  plotting wants a 2-D array, so we squeeze it away.
- `.astype("float32")` — the pixel values are stored as **integers**
  (reflectance × 10 000). Index math needs floats: always cast before dividing.

In [ ]:
BUCKET = "https://a3s.fi/icat3370-2020353-vaasa"

red = rioxarray.open_rasterio(f"{BUCKET}/vaasa_20260716_red.tif").squeeze().astype("float32")
nir = rioxarray.open_rasterio(f"{BUCKET}/vaasa_20260716_nir.tif").squeeze().astype("float32")

ndvi = (nir - red) / (nir + red)
print("NDVI range:", round(float(ndvi.min()), 3), "to", round(float(ndvi.max()), 3))

ndvi.plot.imshow(cmap="RdYlGn", vmin=-1, vmax=1, figsize=(7, 7))
plt.title("NDVI — bands streamed from Allas over HTTPS")
plt.show()

**Reading the map:** water is strongly negative (NIR is absorbed by water),
July forests push toward +0.9, and the mottled area on the right is the city of Vaasa —
mixed vegetation, roofs and streets.

## Route 3 — STAC: finding data in a global archive

Routes 1 and 2 read files whose names we already know. But how was the Vaasa scene
*found* in the first place? Through a **STAC catalog** (SpatioTemporal Asset Catalog) —
a standard, searchable index of satellite archives. You query by area, time and
properties (like cloud cover), and get back items whose *assets* are URLs to COGs —
which you then open exactly like in Route 2.

Below we search the same global archive the course scene came from.
(Finland also runs its own catalog, **Paituli STAC**, with national mosaics —
we will use it in a later exercise.)

In [ ]:
from pystac_client import Client

cat = Client.open("https://earth-search.aws.element84.com/v1")
search = cat.search(
    collections=["sentinel-2-l2a"],
    bbox=[21.4, 62.95, 21.8, 63.15],          # lon/lat around Vaasa
    datetime="2026-06-01/2026-08-18",
    query={"eo:cloud_cover": {"lt": 5}},
    max_items=5,
)
for it in search.items():
    print(f"{it.id}   cloud {it.properties['eo:cloud_cover']:.1f} %   {it.datetime.date()}")

In [ ]:
# open one band of the newest hit at a coarse overview — a whole tile in a few MB
item = next(search.items())
import rasterio
href = item.assets["red"].href
with rasterio.open(href) as src:
    n_ov = len(src.overviews(1))
coarse = rioxarray.open_rasterio(href, overview_level=n_ov - 1)
print("tile at coarsest overview:", coarse.shape)

## Access rights — what you can and cannot do

As a member of the course project you have real access to shared infrastructure.
The rules are simple:

- The **shared data on `/scratch`** is read-only for you — you can open it, not change it.
- The **Allas storage** is shared by the whole project. Do **not** upload to, modify,
  delete from, or change access settings of the course data buckets.
  Every object-storage operation is performed with *your personal credentials*
  and is logged and attributable.
- Save your own outputs in **your home directory** (or `my-work`) — your home has
  a personal quota (~15 GB), so keep saved files small: cropped rasters and figures,
  never whole scenes. Big data is *read* from shared storage or streamed, not copied.

This is exactly how operational EO teams work: shared read-optimized archives,
personal workspaces for results, and attribution for every write.

## Check yourself

1. Change the Route 2 code to compute NDVI **from the `/scratch` copies** instead.
   Do you get the same range?
2. In Route 3, change the bounding box to your home town and find its most recent
   cloud-free Sentinel-2 scene.
3. Why does `.astype("float32")` matter? Remove it and observe what happens to the values.

When you're comfortable with all three routes, you're ready for **Exercise 1**.